In [ ]:
%%writefile fim3d_2.cu
#include <cstdio>          
#include <cstdlib>         
#include <cuda_runtime.h>
#include <vector>          // std::vector
#include <algorithm>       // std::min
#include <chrono>          // tempi sulla CPU
#include <fstream>         // salvataggio su file
const float INF = 1e30f;
#define DIMBLOCK 8          // dimensione del blocco CUDA (8x8x8 = 512 thread)
#define THREAD_LIST 256      // numero di thread per la costruzione della lista A

#define CUDA_CHECK(call)                                                     \
    do {                                                                     \
        cudaError_t err = (call);                                            \
        if (err != cudaSuccess) {                                            \
            printf("Errore CUDA %s:%d: %s\n", __FILE__, __LINE__,            \
                   cudaGetErrorString(err));                                 \
            exit(1);                                                         \
        }                                                                    \
    } while (0)

__device__ float solve_quadratic(float a, float b, float c, float F, float h)
{
    float Fh = F * h;
    if (a > b) { float t = a; a = b; b = t; }      // ordina: a <= b <= c
    if (b > c) { float t = b; b = c; c = t; }
    if (a > b) { float t = a; a = b; b = t; }

    float u = a + Fh;                              // 1 vicino (caso 1)
    if (u <= b) return u;
    u = 0.5f * (a + b + sqrtf(2.0f * Fh * Fh - (a - b) * (a - b)));   // 2 vicini (caso 2 = Alg. 2)
    if (u <= c) return u;
    float d = 3.0f * Fh * Fh - ((a - b) * (a - b) + (b - c) * (b - c) + (c - a) * (c - a));
    return (a + b + c + sqrtf(d)) / 3.0f;          // 3 vicini (caso 3)                                  
}
__device__ float solve_in_block(const float s[BLOCK + 2][BLOCK + 2][BLOCK + 2], int z, int y, int x, float F, float h)
{
    float a = fminf(s[z][y][x - 1], s[z][y][x + 1]);               // a <- min dei vicini lungo x
    float b = fminf(s[z][y - 1][x], s[z][y + 1][x]);               // b <- min dei vicini lungo y
    float c = fminf(s[z - 1][y][x], s[z + 1][y][x]);               // c <- min dei vicini lungo z
    return fminf(s[z][y][x], solve_quadratic3(a, b, c, F, h));      // return min(u_ijk, u)
}
__device__ void process_block(int block, int niter, float* u, const float* F,
                             int N, int nbx, float h, int src, float eps, unsigned char* Cb_out)
{
    __shared__ float s_u[DIMBLOCK + 2][DIMBLOCK + 2][DIMBLOCK + 2];                // block + halo, nella shared memory
    __shared__ unsigned char s_c[DIMBLOCK * DIMBLOCK * DIMBLOCK];                  // i flag Cn dei 512 nodi

    int bi = (block / nbx) % nbx, bj = block % nbx, bk = block / (nbx * nbx);            // riga, colonna e strato del blocco
    int tx = threadIdx.x, ty = threadIdx.y, tz = threadIdx.z;
    int tid = (tz * DIMBLOCK + ty) * DIMBLOCK + tx;                                       // numero del thread nel blocco: 0..511
    int i = bi * DIMBLOCK + ty, j = bj * DIMBLOCK + tx, k = bk * DIMBLOCK + tz;              // nodo di questo thread
    int idx = (k * N + i) * N + j;                                                  // posizione del nodo nella griglia
    bool inside = (i < N && j < N && k < N);                          // l'ultimo blocco puo' sporgere
    bool fixed  = !inside || (idx == src);             // nodi da non aggiornare
    float f = inside ? F[idx] : 1.0f;                  // F_ijk del nodo

    // Global -> Shared: 1000 celle (block + halo), 512 thread
    for (int m = tid; m < (DIMBLOCK + 2) * (DIMBLOCK + 2) * (DIMBLOCK + 2); m += DIMBLOCK * DIMBLOCK * DIMBLOCK) {
        int sz = m / ((DIMBLOCK + 2) * (DIMBLOCK + 2));
        int sy = (m / (DIMBLOCK + 2)) % (DIMBLOCK + 2);
        int sx = m % (DIMBLOCK + 2);                             // cella (sz, sy, sx) della shared memory
        int gi = bi * DIMBLOCK + sy - 1;                         // nodo della griglia corrispondente:
        int gj = bj * DIMBLOCK + sx - 1;                         
        int gk = bk * DIMBLOCK + sz - 1;                         // -1 perche' lo strato/riga/colonna 0 e' l'halo
        bool in_domain = (gi >= 0 && gi < N && gj >= 0 && gj < N && gk >= 0 && gk < N);
        s_u[sz][sy][sx] = in_domain ? u[(gk * N + gi) * N + gj] : INF;      // fuori dal dominio: INF
    }
    __syncthreads();                                         // la block e' completa prima di usarla

    // niter iterazioni locali in shared memory
    bool conv = true;                                        // Cn del nodo
    for (int it = 0; it < niter; ++it) {
        float p = s_u[tz+1][ty + 1][tx + 1];                 // valore prima dell'aggiornamento
        float q = fixed ? p                                  // sorgente / fuori: non cambia
                        : solve_in_block(s_u, tz+1, ty + 1, tx + 1, f, h);
        __syncthreads();                                     // (A) tutti hanno letto i vicini...
        s_u[tz+1][ty + 1][tx + 1] = q;
        __syncthreads();                                     // (B) ...e tutti hanno scritto
        conv = fabsf(p - q) < eps;                           // Cn: il nodo non e' cambiato
    }

    // Shared -> Global: ogni thread riscrive il PROPRIO nodo (non l'halo)
    if (inside) u[idx] = s_u[tz+1][ty + 1][tx + 1];

    // Cb(b) <- reduction(Cn(b)): AND dei 512 flag
    s_c[tid] = conv;
    __syncthreads();
    for (int stride = DIMBLOCK * DIMBLOCK * DIMBLOCK / 2; stride > 0; stride /= 2) {     // 256, ..., 4, 2, 1
        if (tid < stride) s_c[tid] = s_c[tid] && s_c[tid + stride];
        __syncthreads();
    }
    if (tid == 0) Cb_out[block] = s_c[0];                     // vero solo se TUTTI i nodi sono convergenti
}

__global__ void update_active_blocks(const int* list, float* u, const float* F,
                                    int N, int nbx, float h, int src, int niter,
                                    float eps, unsigned char* Cb1)
{
    process_block(list[blockIdx.x], niter, u, F, N, nbx, h, src, eps, Cb1);
}

__global__ void build_active_list(const unsigned char* Cb, unsigned char* active,
                                  int* list, int* count, int n_blocks) 
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // numero del blocco
    if (t >= n_blocks) return;                               // thread in eccesso
    bool act = !Cb[t];                                       // attivo se NON convergente
    active[t] = act;
    if (act) list[atomicAdd(count, 1)] = t;                  // aggiungi t alla lista A e aumenta il contatore
}
__global__ void build_candidate_list(const int* list, int count, int nbx, const unsigned char* active,
                                     const unsigned char* Cb1, unsigned char* Cb,
                                     int* cand, int* cand_list, int* cand_count)
{
    int t = blockIdx.x * blockDim.x + threadIdx.x;           // blockDim = 256
    if (t >= count) return;                                  // thread in eccesso
    int b = list[t];                                         // un blocco della lista A
    Cb[b] = Cb1[b];                                          // non convergente (Caso 1): resta in A
    if (!Cb1[b]) return;                                     // proseguono solo i blocchi appena convergenti

    int bk = b / (nbx * nbx), bi = (b / nbx) % nbx, bj = b % nbx;                          // strato, riga e colonna del blocco
    int  nb[6] = { b - 1,  b + 1,       b - nbx, b + nbx,  b - nbx * nbx, b + nbx * nbx};  // sinistra, destra, sotto, sopra e davanti, dietro
    bool ok[6] = { bj > 0, bj < nbx - 1, bi > 0,  bi < nbx - 1, bk > 0, bk < nbx - 1 };
    for (int d = 0; d < 6; ++d) {
        if (!ok[d]) continue;                                // fuori dalla griglia di blocchi
        if (active[nb[d]] && !Cb1[nb[d]]) continue;          // Caso 1: lo aggiorna già lo Step 1
        if (atomicExch(&cand[nb[d]], 1) == 0)                // primo a segnarlo: niente doppioni
            cand_list[atomicAdd(cand_count, 1)] = nb[d];
    }
}

__global__ void update_candidate_blocks(const int* cand_list, const int* cand_count, int* cand,
                                        float* u, const float* F, int N, int nbx, float h,
                                        int src, float eps, unsigned char* Cb)
{
    if (blockIdx.x >= *cand_count) return;                   // blocco CUDA in eccesso
    int b = cand_list[blockIdx.x];
    process_block(b, 1, u, F, N, nbx, h, src, eps, Cb);      // test del blocco: 1 iterazione
    if (threadIdx.x == 0 && threadIdx.y == 0 && threadIdx.z == 0) cand[b] = 0;   // pronto per il giro successivo
}


int block_fim(std::vector<float>& u, const std::vector<float>& F, int N, float h,
              int src, float eps, int niter, float* ms)
{
    // Blocchi di nodi: nbx per lato, n_blocks in totale
    const int nbx = (N + DIMBLOCK - 1) / DIMBLOCK, n_blocks = nbx * nbx * nbx;
    const size_t bytes = (size_t)N * N * N * sizeof(float);
    
    std::vector<unsigned char> Cb0(n_blocks, 1); // flag Cb iniziali: 1 = convergente, 0 = attivo

    const int sk = src / (N * N), si = (src / N) % N, sj = src % N;  // coordinate del nodo sorgente
    Cb0[(sk / DIMBLOCK) * nbx * nbx + (si / DIMBLOCK) * nbx + (sj / DIMBLOCK)] = 0;  // solo il blocco della sorgente parte attivo

    // Alloca memoria sulla GPU
    float *d_u, *d_F; unsigned char *d_Cb, *d_Cb1, *d_active; int *d_list, *d_count;
    int *d_cand, *d_cand_list, *d_cand_count;
    CUDA_CHECK(cudaMalloc(&d_u, bytes)); CUDA_CHECK(cudaMalloc(&d_F, bytes)); CUDA_CHECK(cudaMalloc(&d_Cb, n_blocks));
    CUDA_CHECK(cudaMalloc(&d_Cb1, n_blocks)); CUDA_CHECK(cudaMalloc(&d_active, n_blocks)); CUDA_CHECK(cudaMalloc(&d_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_count, sizeof(int))); CUDA_CHECK(cudaMalloc(&d_cand, n_blocks * sizeof(int))); CUDA_CHECK(cudaMalloc(&d_cand_list, n_blocks * sizeof(int)));
    CUDA_CHECK(cudaMalloc(&d_cand_count, sizeof(int)));
    // Copia i dati dalla CPU alla GPU
    CUDA_CHECK(cudaMemcpy(d_u, u.data(), bytes, cudaMemcpyHostToDevice)); CUDA_CHECK(cudaMemcpy(d_F, F.data(), bytes, cudaMemcpyHostToDevice));
    CUDA_CHECK(cudaMemcpy(d_Cb, Cb0.data(), n_blocks, cudaMemcpyHostToDevice)); CUDA_CHECK(cudaMemset(d_Cb1, 0, n_blocks));
    CUDA_CHECK(cudaMemset(d_cand, 0, n_blocks * sizeof(int)));

    // Thread per costruire la lista A: uno per blocco di nodi, a gruppi di THREAD_LIST
    const int list_blocks = (n_blocks + THREAD_LIST - 1) / THREAD_LIST;  // numero di gruppi

    cudaEvent_t start, stop;
    CUDA_CHECK(cudaEventCreate(&start));
    CUDA_CHECK(cudaEventCreate(&stop));
    CUDA_CHECK(cudaEventRecord(start));                      // "segna l'ora" prima del ciclo
    int iter = 0;
    while (true){
        int count = 0;
        CUDA_CHECK(cudaMemset(d_count, 0, sizeof(int)));    // azzera il contatore dei blocchi attivi
        build_active_list<<<list_blocks, THREAD_LIST>>>(d_Cb, d_active, d_list, d_count, n_blocks);  // Step 3: lista A dei blocchi attivi (Cb = 0)
        CUDA_CHECK(cudaGetLastError());
        CUDA_CHECK(cudaMemcpy(&count, d_count, sizeof(int), cudaMemcpyDeviceToHost));   // la CPU riceve quanti sono
        if (count == 0) break;                              // lista A vuota: tutto convergente, fine
        iter++;

        // Step 1: ogni blocco di A aggiorna i suoi nodi (niter iterazioni); convergenza in d_Cb1
        update_active_blocks<<<count, dim3(DIMBLOCK, DIMBLOCK, DIMBLOCK)>>>(d_list, d_u, d_F, N, nbx, h, src, niter, eps, d_Cb1); 
        CUDA_CHECK(cudaGetLastError());

        // Step 2a: aggiorna Cb dei blocchi di A e raccoglie i vicini dei blocchi appena convergenti
        CUDA_CHECK(cudaMemset(d_cand_count, 0, sizeof(int)));    // azzera il contatore dei candidati
        build_candidate_list<<<(count + THREAD_LIST - 1) / THREAD_LIST, THREAD_LIST>>>(d_list, count, nbx, d_active, d_Cb1, d_Cb,
                                                                                  d_cand, d_cand_list, d_cand_count);
        CUDA_CHECK(cudaGetLastError());

        // Step 2b: prova ogni candidato con 1 iterazione; se cambia, torna attivo (Cb = 0)
        update_candidate_blocks<<<std::min(6 * count, n_blocks), dim3(DIMBLOCK, DIMBLOCK, DIMBLOCK)>>>(d_cand_list, d_cand_count, d_cand,
                                                                                            d_u, d_F, N, nbx, h, src, eps, d_Cb);
        CUDA_CHECK(cudaGetLastError());

    }
    CUDA_CHECK(cudaEventRecord(stop));                       // "segna l'ora" dopo il ciclo
    CUDA_CHECK(cudaEventSynchronize(stop));                  // aspetta che la GPU ci arrivi
    CUDA_CHECK(cudaEventElapsedTime(ms, start, stop));       // tempo trascorso [ms]
    CUDA_CHECK(cudaEventDestroy(start));CUDA_CHECK(cudaEventDestroy(stop));
    
    // Risultato sull'host e pulizia
    CUDA_CHECK(cudaMemcpy(u.data(), d_u, bytes, cudaMemcpyDeviceToHost)); CUDA_CHECK(cudaFree(d_u)); CUDA_CHECK(cudaFree(d_F));CUDA_CHECK(cudaFree(d_Cb));
    CUDA_CHECK(cudaFree(d_Cb1)); CUDA_CHECK(cudaFree(d_active)); CUDA_CHECK(cudaFree(d_list)); CUDA_CHECK(cudaFree(d_count));
    CUDA_CHECK(cudaFree(d_cand)); CUDA_CHECK(cudaFree(d_cand_list)); CUDA_CHECK(cudaFree(d_cand_count));
    return iter;

}
int main(){

}